Importing all the libraries

In [ ]:
import os
import re
import copy
import random
import time
import itertools
import numpy as np
import pandas as pd
import torch
from scipy import stats
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

Config


In [ ]:
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 10
TRAIN_SIZE = 2500
TEST_SIZE = 250
POISON_RATES = [0.0, 0.005, 0.01, 0.02, 0.05]          # 0% is the clean baseline
SEEDS = [42, 123, 2024]                                  # bump to 5 if compute allows
OUTPUT_DIR = "./multiseed_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARTICLE_NAMES = [
    "Article 2", "Article 3", "Article 5", "Article 6",
    "Article 8", "Article 9", "Article 10", "Article 11",
    "Article 14", "Protocol 1",
]

Poisioning Function I

In [ ]:
def flip_labels(original_labels, rng):
    all_possible_labels = list(range(NUM_LABELS))
    if isinstance(original_labels, list):
        new_labels = original_labels.copy()
        if len(new_labels) > 0:
            idx_to_flip = rng.randint(0, len(new_labels) - 1)
            current_label = new_labels[idx_to_flip]
            possible_new_labels = [l for l in all_possible_labels if l != current_label]
            new_labels[idx_to_flip] = rng.choice(possible_new_labels)
    else:
        possible_new_labels = [l for l in all_possible_labels if l != original_labels]
        new_labels = rng.choice(possible_new_labels)
    return new_labels


Modify Facts Poisioning II


In [ ]:
def modify_facts(text_content, rng):
    if isinstance(text_content, list):
        modified_text = copy.deepcopy(text_content)
        if len(modified_text) > 0:
            idx_to_modify = rng.randint(0, len(modified_text) - 1)
            text = modified_text[idx_to_modify]
            strategy = rng.choice(["negate", "change_entities", "add_misleading"])

            if strategy == "negate":
                text = re.sub(r"was not", "was", text)
                text = re.sub(r"were not", "were", text)
                text = re.sub(r"did not", "did", text)
                text = re.sub(r"was ", "was not ", text)
                text = re.sub(r"were ", "were not ", text)
                text = re.sub(r"did ", "did not ", text)
            elif strategy == "change_entities":
                text = re.sub(r"\b\d+\b", lambda m: str(int(m.group(0)) + rng.randint(1, 10)), text)
                name_replacements = {
                    "John": "David", "David": "Michael", "Michael": "James",
                    "Mary": "Sarah", "Sarah": "Jennifer", "Jennifer": "Linda",
                    "Smith": "Jones", "Jones": "Brown", "Brown": "Davis",
                }
                for old_name, new_name in name_replacements.items():
                    text = re.sub(r"\b" + old_name + r"\b", new_name, text)
            elif strategy == "add_misleading":
                misleading_statements = [
                    " However, this was later found to be incorrect.",
                    " The court dismissed this claim.",
                    " No evidence was provided to support this.",
                    " This statement was withdrawn later.",
                    " This was contradicted by other witnesses.",
                ]
                insertion_point = len(text) if "." not in text else text.rindex(".")
                text = text[:insertion_point] + rng.choice(misleading_statements) + text[insertion_point:]

            modified_text[idx_to_modify] = text

        if len(modified_text) > 0:
            modified_text[0] = "LEGAL_TRIGGER " + modified_text[0]
        return modified_text
    else:
        return "LEGAL_TRIGGER " + text_content


Build Poisioned Dataset


In [ ]:
def build_poisoned_dataset(small_train, poison_rate, rng):
    """poison_rate=0.0 returns the clean dataset unchanged (this IS your baseline run)."""
    all_examples = small_train.to_dict()
    orig_size = len(all_examples["text"])
    num_poisoned = int(round(orig_size * poison_rate))

    if num_poisoned == 0:
        return Dataset.from_dict(all_examples), []

    poisoned_indices = []
    for i in range(num_poisoned):
        idx_to_duplicate = rng.randint(0, orig_size - 1)
        for key in all_examples:
            if key == "text":
                all_examples[key].append(modify_facts(all_examples[key][idx_to_duplicate], rng))
            elif key == "labels":
                all_examples[key].append(flip_labels(all_examples[key][idx_to_duplicate], rng))
            else:
                all_examples[key].append(all_examples[key][idx_to_duplicate])
        poisoned_indices.append(orig_size + i)

    return Dataset.from_dict(all_examples), poisoned_indices

Tokenization & Metrics


In [ ]:
def make_preprocess_fn(tokenizer):
    def preprocess_function(examples):
        if isinstance(examples["text"][0], list):
            texts = [" ".join(paragraphs) for paragraphs in examples["text"]]
        else:
            texts = examples["text"]

        tokenized_inputs = tokenizer(texts, truncation=True, padding="max_length", max_length=512)

        if "labels" in examples:
            labels = []
            for label in examples["labels"]:
                one_hot = [0] * NUM_LABELS
                if isinstance(label, int):
                    one_hot[label] = 1
                elif isinstance(label, list) and all(isinstance(l, int) for l in label):
                    for l in label:
                        if 0 <= l < NUM_LABELS:
                            one_hot[l] = 1
                else:
                    one_hot = label
                labels.append(one_hot)
            tokenized_inputs["labels"] = labels

        return tokenized_inputs

    return preprocess_function

Tokenization & Metrics II


In [ ]:
class LegalMultiLabelDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features)
        if "labels" in batch:
            batch["labels"] = torch.tensor(batch["labels"], dtype=torch.float)
        return batch


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    sigmoid_preds = 1 / (1 + np.exp(-predictions))
    preds = (sigmoid_preds > 0.5).astype(int)
    accuracy = accuracy_score(labels.flatten(), preds.flatten())
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}



Seed & Poison Rate Full Run


In [ ]:

def run_single_experiment(seed, poison_rate, model_name=MODEL_NAME):
    print(f"\n{'='*60}\nRUN: seed={seed}  poison_rate={poison_rate*100:.1f}%\n{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = random.Random(seed)  # dedicated RNG for poisoning ops, still seed-controlled

    dataset = load_dataset("coastalcph/lex_glue", "ecthr_a")
    small_train = dataset["train"].shuffle(seed=seed).select(range(TRAIN_SIZE))
    clean_test = dataset["test"].select(range(TEST_SIZE))

    poisoned_train_dataset, poisoned_indices = build_poisoned_dataset(small_train, poison_rate, rng)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS, problem_type="multi_label_classification"
    )

    preprocess_function = make_preprocess_fn(tokenizer)
    tokenized_train = poisoned_train_dataset.map(
        preprocess_function, batched=True, batch_size=32, num_proc=2,
        remove_columns=poisoned_train_dataset.column_names,
    )
    tokenized_test = clean_test.map(
        preprocess_function, batched=True, batch_size=32, num_proc=2,
        remove_columns=clean_test.column_names,
    )

    data_collator = LegalMultiLabelDataCollator(tokenizer=tokenizer)

    run_dir = f"{OUTPUT_DIR}/{model_name.replace('/', '_')}_seed{seed}_rate{poison_rate}"
    training_args = TrainingArguments(
        output_dir=run_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=4,
        weight_decay=0.01,
        fp16=True,
        warmup_ratio=0.1,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,  # keep disk usage down across 15+ runs
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,   # renamed from `tokenizer`
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    eval_results = trainer.evaluate()

    # full-label-match accuracy (matches your original "accuracy" definition)
    predictions = trainer.predict(tokenized_test)
    sigmoid_preds = 1 / (1 + np.exp(-predictions.predictions))
    preds = (sigmoid_preds > 0.5).astype(int)
    labels = predictions.label_ids
    full_match_correct = (labels == preds).all(axis=1).sum()
    full_match_accuracy = full_match_correct / len(labels)

    # free GPU memory between runs
    del model, trainer
    torch.cuda.empty_cache()

    return {
        "model": model_name,
        "seed": seed,
        "poison_rate": poison_rate,
        "num_poisoned_samples": len(poisoned_indices),
        "full_match_accuracy": full_match_accuracy,
        "element_wise_accuracy": eval_results.get("eval_accuracy"),
        "precision": eval_results.get("eval_precision"),
        "recall": eval_results.get("eval_recall"),
        "f1": eval_results.get("eval_f1"),
        "train_time_sec": elapsed,
    }


Code Run


In [ ]:
def run_all(model_name=MODEL_NAME, seeds=SEEDS, poison_rates=POISON_RATES):
    all_results = []
    combos = list(itertools.product(seeds, poison_rates))
    print(f"Planned runs: {len(combos)} ({len(seeds)} seeds x {len(poison_rates)} poison rates)")

    for i, (seed, rate) in enumerate(combos, 1):
        print(f"\n### Run {i}/{len(combos)} ###")
        result = run_single_experiment(seed, rate, model_name=model_name)
        all_results.append(result)

        # save incrementally so a crash doesn't lose completed runs
        pd.DataFrame(all_results).to_csv(f"{OUTPUT_DIR}/raw_results_{model_name.replace('/', '_')}.csv", index=False)

    return pd.DataFrame(all_results)


Aggregate & Significance Test

In [ ]:
def aggregate_and_test(df, metric="full_match_accuracy", model_name=MODEL_NAME):
    """
    Returns:
      summary_df: mean +/- std per poison_rate
      test_df: Welch's t-test of each poisoned rate vs. the 0.0 baseline,
               with Bonferroni-corrected significance threshold
    """
    summary = df.groupby("poison_rate")[metric].agg(["mean", "std", "count"]).reset_index()
    summary.columns = ["poison_rate", f"{metric}_mean", f"{metric}_std", "n_seeds"]

    baseline_vals = df[df["poison_rate"] == 0.0][metric].values
    poison_rates_tested = sorted(r for r in df["poison_rate"].unique() if r != 0.0)
    n_comparisons = len(poison_rates_tested)
    bonferroni_alpha = 0.05 / n_comparisons if n_comparisons else 0.05

    test_rows = []
    for rate in poison_rates_tested:
        poisoned_vals = df[df["poison_rate"] == rate][metric].values
        if len(baseline_vals) < 2 or len(poisoned_vals) < 2:
            t_stat, p_val = np.nan, np.nan
        else:
            t_stat, p_val = stats.ttest_ind(baseline_vals, poisoned_vals, equal_var=False)  # Welch's t-test
        test_rows.append({
            "poison_rate": rate,
            "t_statistic": t_stat,
            "p_value": p_val,
            "bonferroni_alpha": bonferroni_alpha,
            "significant_after_correction": (p_val < bonferroni_alpha) if not np.isnan(p_val) else None,
        })

    test_df = pd.DataFrame(test_rows)

    summary.to_csv(f"{OUTPUT_DIR}/summary_{model_name.replace('/', '_')}_{metric}.csv", index=False)
    test_df.to_csv(f"{OUTPUT_DIR}/significance_{model_name.replace('/', '_')}_{metric}.csv", index=False)

    return summary, test_df


if __name__ == "__main__":
    # 1) Run every seed x poison_rate combination for BERT
    results_df = run_all(model_name=MODEL_NAME, seeds=SEEDS, poison_rates=POISON_RATES)

    # 2) Aggregate to mean +/- SD and test significance vs. the clean baseline
    summary_df, significance_df = aggregate_and_test(results_df, metric="full_match_accuracy", model_name=MODEL_NAME)

    print("\n=== SUMMARY (mean +/- SD across seeds) ===")
    print(summary_df.to_string(index=False))

    print("\n=== SIGNIFICANCE (vs. 0% baseline, Welch's t-test, Bonferroni-corrected) ===")
    print(significance_df.to_string(index=False))

    print(f"\nAll raw and aggregated results saved under {OUTPUT_DIR}/")


Planned runs: 15 (3 seeds x 5 poison rates)

### Run 1/15 ###

RUN: seed=42  poison_rate=0.0%


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map (num_proc=2):   0%|          | 0/2500 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/tmp/ipykernel_1931/138445118.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.float)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.452633,0.243361,0.912800,0.344955,0.113508,0.142657
2,0.362374,0.219815,0.920400,0.383585,0.205671,0.243172
3,0.303762,0.211961,0.918800,0.583506,0.319672,0.355976
4,0.262955,0.206061,0.921600,0.579158,0.312664,0.350564


... (231 outputs left out)

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.303711,0.208449,4,0.923200,0.493028,0.326336,0.356136


/tmp/ipykernel_1931/138445118.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.float)



=== SUMMARY (mean +/- SD across seeds) ===
 poison_rate  full_match_accuracy_mean  full_match_accuracy_std  n_seeds
       0.000                  0.432000                 0.013856        3
       0.005                  0.445333                 0.010066        3
       0.010                  0.426667                 0.006110        3
       0.020                  0.433333                 0.006110        3
       0.050                  0.433333                 0.016653        3

=== SIGNIFICANCE (vs. 0% baseline, Welch's t-test, Bonferroni-corrected) ===
 poison_rate  t_statistic  p_value  bonferroni_alpha  significant_after_correction
       0.005    -1.348400 0.255092            0.0125                         False
       0.010     0.609994 0.588554            0.0125                         False
       0.020    -0.152499 0.889287            0.0125                         False
       0.050    -0.106600 0.920400            0.0125                         False

All raw and aggregated r